# CS781 Legal NER — Self-contained new experiments

This notebook reconstructs the known champion + rules pipeline, checks the established validation hybrid score (~0.333307), then runs **two new model experiments** from independent copies of the champion:

1. **Experiment A:** class-weighted cross-entropy fine-tuning.
2. **Experiment B:** class-weighted focal-loss fine-tuning (gamma=2).

Both train on fold 1 and evaluate on fold 2 using exact-span macro-F1 and the existing counsel/judge/WIT rules plus the selected CASENO+STAT boundary correction. No test gold labels are used. Existing champion files and predictions are never overwritten.

Run the notebook top-to-bottom. If the baseline reproduction check fails, training stops rather than producing misleading results.


In [ ]:
# 1. Necessary imports only + runtime
import os, json, re, gc, math, copy
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
from tokenizers import Tokenizer
from transformers import GPT2Config, GPT2Model
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Inventory all available datasets/checkpoints before choosing anything
ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]
CHECKPOINT_NAMES = {
    "champion_recovery.pt", "best_ner_model.pt", "checkpoint_latest.pt",
    "targeted_pilot_8_steps.pt", "best_model.pt", "model.pt"
}
DATASET_NAMES = {
    "legal_bpe_32k.json", "fold1-text.json", "fold1-gold.json",
    "fold2-text.json", "fold2-gold.json", "fold3-text.json",
    "train.json", "test.json"
}
all_files = []
for root in ROOTS:
    if root.exists():
        all_files.extend(p for p in root.rglob("*") if p.is_file())

all_files = sorted(set(all_files))
checkpoint_files = [p for p in all_files if p.name in CHECKPOINT_NAMES or p.suffix.lower() in {".pt", ".pth"}]
dataset_files = [p for p in all_files if p.name in DATASET_NAMES or ("fold" in p.name.lower() and p.suffix.lower() == ".json")]

print("=== CHECKPOINT INVENTORY ===")
if checkpoint_files:
    for p in checkpoint_files:
        print(f"{p}  [{p.stat().st_size / 1024**2:.1f} MiB]")
else:
    print("No checkpoint files found.")

print("\n=== DATASET/TOKENIZER INVENTORY ===")
for p in dataset_files:
    print(p)

print("\n=== BACKUP DATASET DIRECTORIES ===")
for p in sorted({str(x.parent) for x in dataset_files}):
    print(p)

# Pick the recovery champion explicitly; don't accidentally choose an arbitrary checkpoint_latest.pt.
champion_candidates = [p for p in checkpoint_files if p.name == "champion_recovery.pt"]
if not champion_candidates:
    champion_candidates = [p for p in checkpoint_files if p.name == "best_ner_model.pt"]
if not champion_candidates:
    raise FileNotFoundError(
        "No champion_recovery.pt or best_ner_model.pt found under /kaggle/input or /kaggle/working. "
        "Attach the CS781 recovery/checkpoint dataset in Kaggle."
    )

def rank_champion_path(p):
    s = str(p).lower()
    return (
        0 if "cs781_recovery" in s else 1,
        0 if "working-backup" in s else 1,
        0 if "/kaggle/input/" in s else 1,
        s
    )

CHAMPION_PATH = sorted(champion_candidates, key=rank_champion_path)[0]

phase1_candidates = [
    p for p in checkpoint_files
    if p.name == "checkpoint_latest.pt" and "cs781-phase-1-checkpoints" in str(p)
]
if not phase1_candidates:
    phase1_candidates = [
        p for p in checkpoint_files
        if p.name == "checkpoint_latest.pt" and "phase-1" in str(p).lower()
    ]
if not phase1_candidates:
    phase1_candidates = [p for p in checkpoint_files if p.name == "checkpoint_latest.pt"]

if not phase1_candidates:
    raise FileNotFoundError("No Phase-1 checkpoint_latest.pt found.")

PHASE1_PATH = sorted(phase1_candidates, key=lambda p: (0 if "/kaggle/input/" in str(p) else 1, str(p)))[0]

print("\nSELECTED NER CHAMPION:", CHAMPION_PATH)
print("SELECTED PHASE-1 ENCODER:", PHASE1_PATH)
print("Selection policy: recovery champion for NER; Phase-1 checkpoint for encoder.")

In [ ]:
# 3. Inspect checkpoint metadata; only the known champion is selected for NER inference
def safe_torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

recovery = safe_torch_load(CHAMPION_PATH)
phase1_ckpt = safe_torch_load(PHASE1_PATH)

print("Champion checkpoint keys:", list(recovery.keys()) if isinstance(recovery, dict) else type(recovery))
print("Phase-1 checkpoint keys:", list(phase1_ckpt.keys()) if isinstance(phase1_ckpt, dict) else type(phase1_ckpt))
if isinstance(phase1_ckpt, dict):
    print("Phase-1 step/epoch metadata:", {
        k: phase1_ckpt.get(k) for k in ("step", "global_step", "epoch", "best_step")
        if k in phase1_ckpt
    })

# Report other checkpoints' metadata without silently replacing the champion.
print("\n=== OTHER CHECKPOINT METADATA ===")
for p in checkpoint_files:
    if p in (CHAMPION_PATH, PHASE1_PATH):
        continue
    try:
        obj = safe_torch_load(p)
        if isinstance(obj, dict):
            keys = list(obj.keys())
            meta = {k: obj.get(k) for k in ("step", "global_step", "epoch", "optimizer_steps")
                    if k in obj and isinstance(obj.get(k), (int, float, str))}
            state = obj.get("model_state_dict")
            shape_info = None
            if isinstance(state, dict):
                cls = state.get("classifier.weight")
                if cls is not None and hasattr(cls, "shape"):
                    shape_info = tuple(cls.shape)
            print(f"{p}: keys={keys[:12]}, metadata={meta}, classifier.weight={shape_info}")
        else:
            print(f"{p}: object={type(obj).__name__}")
        del obj
    except Exception as e:
        print(f"{p}: could not inspect ({type(e).__name__}: {e})")

In [ ]:
# 4. Reconstruct model and load tokenizer/data files
BIO_LABELS = list(recovery["bio_labels"])
label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for i, label in enumerate(BIO_LABELS)}
ENTITY_LABELS = [
    "APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT",
    "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"
]
assert len(BIO_LABELS) == 25, f"Expected 25 BIO labels, got {len(BIO_LABELS)}"

cfg = phase1_ckpt.get("model_config")
if cfg is None:
    raise KeyError("Phase-1 checkpoint has no model_config; cannot reconstruct encoder safely.")

gpt2_config = GPT2Config(
    vocab_size=cfg["vocab_size"],
    n_positions=cfg["context_length"],
    n_ctx=cfg["context_length"],
    n_embd=cfg["hidden_size"],
    n_layer=cfg["num_layers"],
    n_head=cfg["num_heads"],
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0
)

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(pretrained_lm.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.classifier(self.dropout(outputs.last_hidden_state))

phase1_lm = GPT2Model(gpt2_config)
phase1_state = {
    k.removeprefix("transformer."): v
    for k, v in phase1_ckpt["model_state_dict"].items()
    if k.startswith("transformer.")
}
phase1_lm.load_state_dict(phase1_state, strict=True)
lm_wrapper = type("LMWrapper", (), {"transformer": phase1_lm, "config": gpt2_config})()
champion_model = LegalNERModel(lm_wrapper, len(BIO_LABELS)).to(device)
champion_model.load_state_dict(recovery["model_state_dict"], strict=True)
champion_model.eval()

def find_named_file(filename, preferred_tokens=()):
    matches = [p for p in all_files if p.name == filename]
    if not matches:
        raise FileNotFoundError(f"Required file {filename} not found in Kaggle input/working.")
    preferred = [p for p in matches if all(t.lower() in str(p).lower() for t in preferred_tokens)]
    candidates = preferred or matches
    return sorted(candidates, key=lambda p: (0 if "/kaggle/input/" in str(p) else 1, str(p)))[0]

tokenizer_path = find_named_file("legal_bpe_32k.json")
fold1_text_path = find_named_file("fold1-text.json")
fold1_gold_path = find_named_file("fold1-gold.json")
fold2_text_path = find_named_file("fold2-text.json")
fold2_gold_path = find_named_file("fold2-gold.json")
fold3_text_path = find_named_file("fold3-text.json")

tokenizer = Tokenizer.from_file(str(tokenizer_path))
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_texts = load_json(fold1_text_path)
train_gold = load_json(fold1_gold_path)
val_texts = load_json(fold2_text_path)
val_gold = load_json(fold2_gold_path)
test_texts = load_json(fold3_text_path)

# Normalize document IDs to strings consistently.
train_texts = {str(k): v for k, v in train_texts.items()}
train_gold = {str(k): v for k, v in train_gold.items()}
val_texts = {str(k): v for k, v in val_texts.items()}
val_gold = {str(k): v for k, v in val_gold.items()}
test_texts = {str(k): v for k, v in test_texts.items()}

print("Champion loaded:", CHAMPION_PATH)
print("Tokenizer:", tokenizer_path, "| vocab size:", tokenizer.get_vocab_size())
print("Documents: train", len(train_texts), "| validation", len(val_texts), "| test", len(test_texts))
print("Train/val IDs disjoint:", set(train_texts).isdisjoint(val_texts))
print("Validation text/gold IDs match:", set(val_texts) == set(val_gold))
print("Train text/gold IDs match:", set(train_texts) == set(train_gold))
assert set(train_texts) == set(train_gold)
assert set(val_texts) == set(val_gold)

In [ ]:
# 5. Find and parse champion thresholds
threshold_candidates = [
    p for p in all_files
    if p.suffix.lower() == ".json" and "threshold" in p.name.lower()
]
print("Threshold files found:")
for p in threshold_candidates:
    print(" ", p)

def extract_thresholds(obj):
    if not isinstance(obj, dict):
        return None
    for key in ("champ_thr", "thresholds", "label_thresholds", "threshold_by_label"):
        if isinstance(obj.get(key), dict) and obj[key] and all(isinstance(v, (int, float)) for v in obj[key].values()):
            return {str(k): float(v) for k, v in obj[key].items()}
    if obj and all(isinstance(v, (int, float)) for v in obj.values()):
        return {str(k): float(v) for k, v in obj.items()}
    return None

champ_thr = None
threshold_path_used = None
for p in sorted(threshold_candidates, key=lambda p: (
    0 if "champion_thresholds_recovery" in p.name.lower() else 1,
    0 if "cs781_recovery" in str(p).lower() else 1,
    str(p)
)):
    try:
        obj = load_json(p)
        found = extract_thresholds(obj)
        if found and all(lab in found for lab in ENTITY_LABELS):
            champ_thr, threshold_path_used = found, p
            break
    except Exception:
        pass

if champ_thr is None:
    # These are the documented recovery thresholds from the existing notebook.
    champ_thr = {
        "APP": 0.15, "RESP": 0.45, "A.COUNSEL": 0.45, "R.COUNSEL": 0.45,
        "JUDGE": 0.25, "WIT": 0.45, "AUTH": 0.35, "COURT": 0.45,
        "STAT": 0.25, "PREC": 0.35, "DATE": 0.55, "CASENO": 0.35
    }
    print("WARNING: no usable threshold JSON found; using documented recovery thresholds.")
else:
    print("Loaded thresholds from:", threshold_path_used)
print("Thresholds:", champ_thr)

In [ ]:
# 6. Core inference, BIO decoding, and exact-span evaluation helpers
from bisect import bisect_left, bisect_right

NUM_LABELS = len(BIO_LABELS)
legal_start = np.array([tag == "O" or tag.startswith("B-") for tag in BIO_LABELS], dtype=bool)
legal_transition = np.ones((NUM_LABELS, NUM_LABELS), dtype=bool)
for prev_id, prev_tag in enumerate(BIO_LABELS):
    for curr_id, curr_tag in enumerate(BIO_LABELS):
        if curr_tag.startswith("I-"):
            legal_transition[prev_id, curr_id] = prev_tag in (f"B-{curr_tag[2:]}", f"I-{curr_tag[2:]}")

def bio_viterbi_decode(logits):
    emissions = logits.detach().float().cpu().numpy() if isinstance(logits, torch.Tensor) else np.asarray(logits, dtype=np.float64)
    if emissions.ndim != 2 or emissions.shape[1] != NUM_LABELS:
        raise ValueError(f"Expected [sequence_length, {NUM_LABELS}] emissions, got {emissions.shape}")
    if len(emissions) == 0:
        return []
    emissions = emissions.astype(np.float64, copy=False)
    neg_inf = -1e12
    scores = np.full_like(emissions, neg_inf, dtype=np.float64)
    back = np.zeros(emissions.shape, dtype=np.int32)
    scores[0, legal_start] = emissions[0, legal_start]
    for t in range(1, len(emissions)):
        cand = np.where(legal_transition, scores[t-1][:, None], neg_inf) + emissions[t][None, :]
        back[t] = np.argmax(cand, axis=0)
        scores[t] = cand[back[t], np.arange(NUM_LABELS)]
    last = int(np.argmax(scores[-1]))
    path = [last]
    for t in range(len(emissions)-1, 0, -1):
        path.append(int(back[t, path[-1]]))
    return list(reversed(path))

def decode_spans(text, offsets, pred_ids):
    spans, current_label, current_start, current_end = [], None, None, None
    def close():
        nonlocal current_label, current_start, current_end
        if current_label is not None:
            s, e = current_start, current_end
            while s < e and text[s].isspace(): s += 1
            while e > s and text[e-1].isspace(): e -= 1
            if s < e: spans.append((s, e, current_label))
        current_label = current_start = current_end = None
    for (s, e), pred_id in zip(offsets, pred_ids):
        if e <= s:
            continue
        tag = BIO_LABELS[int(pred_id)]
        if tag == "O":
            close(); continue
        prefix, lab = tag.split("-", 1)
        if prefix == "B" or current_label != lab:
            close(); current_label, current_start, current_end = lab, s, e
        else:
            current_end = e
    close()
    return sorted(set(spans))

DIAG_MAX_LEN, DIAG_STRIDE = 512, 448
@torch.inference_mode()
def diag_doc_probs(model, text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = enc.ids, enc.offsets
    n = len(ids)
    if not n:
        return offsets, torch.zeros((0, len(BIO_LABELS)))
    logit_sum = torch.zeros((n, len(BIO_LABELS)), dtype=torch.float32)
    counts = torch.zeros(n, dtype=torch.float32)
    model.eval()
    for start in range(0, n, DIAG_STRIDE):
        end = min(start + DIAG_MAX_LEN, n)
        x = torch.tensor([ids[start:end]], dtype=torch.long, device=device)
        logits = model(x, torch.ones_like(x))[0].float().cpu()
        logit_sum[start:end] += logits
        counts[start:end] += 1
        if end == n: break
    avg_logits = logit_sum / counts.clamp_min(1).unsqueeze(-1)
    return offsets, torch.softmax(avg_logits, dim=-1)

def predict_texts(model, texts):
    result = {}
    model.eval()
    for doc_id, text in tqdm(texts.items(), desc="Predicting", leave=False):
        offsets, probs = diag_doc_probs(model, text)
        if not len(offsets):
            result[doc_id] = []
            continue
        pred_ids = torch.tensor(bio_viterbi_decode(torch.log(probs.clamp_min(1e-12))), dtype=torch.long)
        starts, ends = [a for a, b in offsets], [b for a, b in offsets]
        spans = []
        for s, e, lab in decode_spans(text, offsets, pred_ids.tolist()):
            ix = [i for i in range(bisect_right(ends, s), bisect_left(starts, e)) if ends[i] > starts[i]]
            conf = float(probs[ix, pred_ids[ix]].mean()) if ix else 0.0
            spans.append((int(s), int(e), lab, conf))
        result[doc_id] = spans
    return result

def apply_thr(raw, thresholds):
    return {d: [(s, e, lab) for s, e, lab, conf in spans if conf >= thresholds.get(lab, 0.0)] for d, spans in raw.items()}

def calculate_exact_span_metrics(gold_by_doc, pred_by_doc, labels=ENTITY_LABELS):
    metrics = {}
    for lab in labels:
        gold_set, pred_set = set(), set()
        for d, ents in gold_by_doc.items():
            for ent in ents:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    gold_set.add((str(d), int(ent[0]), int(ent[1]), lab))
        for d, spans in pred_by_doc.items():
            for ent in spans:
                if len(ent) >= 3 and str(ent[2]) == lab:
                    pred_set.add((str(d), int(ent[0]), int(ent[1]), lab))
        tp = len(gold_set & pred_set); fp = len(pred_set - gold_set); fn = len(gold_set - pred_set)
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2*p*r/(p+r) if p+r else 0.0
        metrics[lab] = {"gold":len(gold_set), "pred":len(pred_set), "TP":tp, "FP":fp, "FN":fn,
                        "precision":p, "recall":r, "f1":f}
    metrics["MACRO_F1"] = sum(metrics[x]["f1"] for x in labels) / len(labels)
    return metrics

# Verify exact recovery weights and a forward pass.
state = recovery["model_state_dict"]
loaded = champion_model.state_dict()
assert set(state) == set(loaded)
max_diff = max((loaded[k].detach().cpu() - state[k]).abs().max().item() for k in state)
assert max_diff == 0.0
with torch.inference_mode():
    z = champion_model(torch.zeros((1, 8), dtype=torch.long, device=device), torch.ones((1, 8), dtype=torch.long, device=device))
assert tuple(z.shape) == (1, 8, len(BIO_LABELS))
print("Champion weights verified exactly; output shape:", tuple(z.shape))

In [ ]:

# CS781: candidate rules for RESP headers and explicit witness identifiers.
# Validation is for evaluation only. No model or file is modified.

import re
from collections import Counter

RESP_LABEL = "RESP"
WIT_LABEL = "WIT"

# Matches common Indian judgment header styles:
# "... Versus NAME ... Respondent(s)"
# "RESPONDENT: NAME"
# "PETITIONER: NAME RESPONDENT: NAME"
RESP_MARKER = re.compile(
    r"\bRESPONDENTS?\b\s*[\(\[]?\s*[Ss]?\s*[\)\]]?\s*[:.]?",
    re.I
)
VERSUS_MARKER = re.compile(r"\b(?:VERSUS|VS\.?|V\.?)\b", re.I)

# Witness / listed-witness identifiers, including LW.
WIT_ID = re.compile(
    r"\b(?:P\s*\.?\s*W|D\s*\.?\s*W|C\s*\.?\s*W|L\s*\.?\s*W)"
    r"\s*[-.]?\s*\d+[A-Z]?\b",
    re.I
)

# Conservative name candidate: 1–5 capitalized name tokens, optionally
# including initials and common prefixes/titles.
NAME_TOKEN = (
    r"(?:[A-Z][A-Za-z.'’\-]*|"
    r"[A-Z]\.)"
)
NAME_CANDIDATE = re.compile(
    rf"(?<![\w])(?:Dr\.?\s+|Shri\s+|Smt\.?\s+|Mr\.?\s+|Mrs\.?\s+)?"
    rf"{NAME_TOKEN}(?:\s+{NAME_TOKEN}){{0,4}}"
)

def _add_candidate(out, text, start, end, label):
    # Trim whitespace and common punctuation without losing source offsets.
    while start < end and (text[start].isspace() or text[start] in " \t:;,."):
        start += 1
    while end > start and (text[end-1].isspace() or text[end-1] in " \t:;,."):
        end -= 1
    if start < end:
        out.add((start, end, label))

def resp_header_candidates(text, header_limit=2500):
    """Candidate spans from respondent header regions only."""
    out = set()
    header = text[:min(len(text), header_limit)]

    # Pattern A: names in the text between Versus and the Respondent marker.
    for vm in VERSUS_MARKER.finditer(header):
        rm = RESP_MARKER.search(header, vm.end())
        if rm is None:
            continue
        # Avoid grabbing a large body section if no nearby respondent label.
        if rm.start() - vm.end() > 700:
            continue

        segment_start, segment_end = vm.end(), rm.start()
        segment = header[segment_start:segment_end]

        # Split on common party separators, preserving absolute offsets.
        chunks = re.split(r"\s+(?:AND|&)\s+|[,;]\s*(?=\d|\b[A-Z])",
                          segment, flags=re.I)
        cursor = 0
        for chunk in chunks:
            local = segment.find(chunk, cursor)
            if local < 0:
                continue
            cursor = local + len(chunk)
            # Only keep name-like chunks, not legal role labels.
            for match in NAME_CANDIDATE.finditer(chunk):
                candidate = match.group(0).strip()
                if candidate.casefold() in {
                    "versus", "appellant", "appellants", "respondent",
                    "respondents", "petitioner", "petitioners"
                }:
                    continue
                s = segment_start + local + match.start()
                e = segment_start + local + match.end()
                _add_candidate(out, text, s, e, RESP_LABEL)

    # Pattern B: explicit "RESPONDENT: NAME" style metadata.
    for rm in re.finditer(r"\bRESPONDENT\b\s*:\s*", header, re.I):
        start = rm.end()
        end = min(len(header), start + 180)
        segment = header[start:end]
        # Stop at the next obvious metadata field or judgment heading.
        stop = re.search(
            r"\b(?:DATE OF JUDGMENT|BENCH|JUDGMENT|J U D G M E N T)\b",
            segment, re.I
        )
        if stop:
            segment = segment[:stop.start()]
        for match in NAME_CANDIDATE.finditer(segment):
            candidate = match.group(0).strip()
            if candidate.casefold() in {"and", "or", "ors", "the"}:
                continue
            _add_candidate(out, text, start + match.start(),
                           start + match.end(), RESP_LABEL)

    return out

def witness_id_candidates(text):
    """Candidate person spans immediately adjacent to witness identifiers."""
    out = set()

    for wm in WIT_ID.finditer(text):
        # Most common form: "Jagdish (PW-5)" or "Dr. X-PW2".
        left_start = max(0, wm.start() - 100)
        left = text[left_start:wm.start()]

        # Restrict to text after a sentence/punctuation boundary when possible.
        boundary_positions = [
            p for p in (left.rfind("."), left.rfind(";"),
                        left.rfind(":"), left.rfind("\n"))
            if p >= 0
        ]
        if boundary_positions:
            left_start += max(boundary_positions) + 1
            left = text[left_start:wm.start()]

        matches = list(NAME_CANDIDATE.finditer(left))
        if matches:
            # Keep the closest name-like expression before the identifier.
            m = matches[-1]
            candidate = m.group(0).strip()
            if candidate.casefold() not in {
                "witness", "statement", "evidence", "examined",
                "cross examined", "and", "the"
            }:
                _add_candidate(out, text, left_start + m.start(),
                               left_start + m.end(), WIT_LABEL)

        # Also handle "PW-5: Jagdish" or "LW-3, NAME".
        right = text[wm.end():min(len(text), wm.end() + 100)]
        right_boundary = re.search(r"[.;\n]", right)
        if right_boundary:
            right = right[:right_boundary.start()]
        m = NAME_CANDIDATE.search(right)
        if m:
            _add_candidate(out, text, wm.end() + m.start(),
                           wm.end() + m.end(), WIT_LABEL)

    return out

def candidates_for_text(text):
    return resp_header_candidates(text) | witness_id_candidates(text)

val_rule_candidates = {
    str(doc_id): sorted(candidates_for_text(text))
    for doc_id, text in val_texts.items()
}
train_rule_candidates = {
    str(doc_id): sorted(candidates_for_text(text))
    for doc_id, text in train_texts.items()
}

print("Candidate generation complete.")
print("Validation documents:", len(val_rule_candidates))
print("Validation RESP candidates:",
      sum(1 for spans in val_rule_candidates.values()
          for _, _, label in spans if label == RESP_LABEL))
print("Validation WIT candidates:",
      sum(1 for spans in val_rule_candidates.values()
          for _, _, label in spans if label == WIT_LABEL))
print("These are candidate rules, not accepted predictions.")


In [ ]:

# CS781 — Restore original rules + evaluate WIT hybrid
# Validation-only. No training, checkpoint writes, or prediction-file writes.

import re
from collections import Counter

_required = [
    "train_texts", "train_gold", "val_texts", "val_gold",
    "champion_model", "champ_thr", "predict_texts", "apply_thr",
    "diag_doc_probs", "bio_viterbi_decode", "decode_spans",
    "calculate_exact_span_metrics", "ENTITY_LABELS",
    "val_rule_candidates",
]
_missing = [x for x in _required if x not in globals()]
if _missing:
    raise RuntimeError("Missing required objects: " + ", ".join(_missing))

# ------------------------------------------------------------
# 1. Original counsel candidate rules from the original notebook
# ------------------------------------------------------------
# Reuse the original function if it exists; otherwise restore its
# conservative section-based candidate extraction.

if "rule_spans" not in globals():
    CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
    INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
    WORD = rf"(?:{INIT}|{CAP})"
    NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"

    SECTION_RE = re.compile(
        r"\b(?:"
        r"counsel\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
        r"applicant|complainant|accused|state|opposite\s+party)s?"
        r"|for\s+(?:the\s+)?(?:appellant|respondent|petitioner|applicant|"
        r"complainant|accused|state|opposite\s+party)s?"
        r"|appearing\s+for"
        r"|agent\s+for\s+(?:the\s+)?(?:appellant|respondent|petitioner|"
        r"applicant|complainant|accused|state|opposite\s+party)s?"
        r")\s*[:\-]?\s*",
        re.I
    )
    SECTION_STOP_RE = re.compile(
        r"\b(?:Hon['’]?ble|JUDGMENT|ORDER|CORAM|"
        r"the\s+following\s+order|judgment\s+was\s+delivered)\b",
        re.I
    )

    def rule_spans(text):
        spans = set()
        for sm in SECTION_RE.finditer(text):
            start = sm.end()
            region = text[start:min(len(text), start + 500)]
            stop = SECTION_STOP_RE.search(region)
            if stop:
                region = region[:stop.start()]
            region = re.split(
                r"\n\s*\n|(?:\s{3,})|\b(?:versus|JUDGMENT|ORDER)\b",
                region, maxsplit=1, flags=re.I
            )[0]

            for m in re.finditer(NAME2, region):
                candidate = m.group(0).strip()
                if candidate.lower() in {
                    "appellant", "respondent", "petitioner", "applicant",
                    "state", "court", "justice", "judge", "counsel",
                    "advocate", "senior counsel", "learned counsel"
                }:
                    continue
                spans.add((start + m.start(), start + m.end()))
        return spans

    print(
        "NOTE: rule_spans was absent. Restored the section-based candidate "
        "extractor from the original notebook's candidate-rule family."
    )

# ------------------------------------------------------------
# 2. Original counsel side assignment and fold-1 policy selection
# ------------------------------------------------------------
PARTY_A = (
    r"(?:petitioners?|appellants?|applicants?|plaintiffs?|"
    r"complainants?|prosecution|revisionists?)"
)
PARTY_R = (
    r"(?:respondents?|defendants?|accused|non-\s*applicants?|"
    r"opposite\s+part(?:y|ies))"
)
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"
CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")
A_, R_ = "A.COUNSEL", "R.COUNSEL"
COUNSEL_ORDER = ["hdr", "right", "left", "inherit"]

def _cues(text, lo, hi):
    return (
        [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
        + [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    )

def _assign_side(text, s, e, prev):
    for step in COUNSEL_ORDER:
        if step == "right":
            cues = _cues(text, e, min(len(text), e + 200))
            if cues:
                return min(cues)[2], "right"
        elif step == "left":
            cues = _cues(text, max(0, s - 300), s)
            cues = [x for x in cues if x[1] <= s]
            if cues:
                return max(cues, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail):
                return A_, "hdr"
            if HDR_R.search(tail):
                return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, source = _assign_side(text, s, e, prev)
        prev = (e, lab, source)
        if source == "default":
            if policy == "drop":
                continue
            if policy == "A":
                lab = A_
        out.append((int(s), int(e), lab))
    return out

def _counsel_fold1_score(policy):
    gold = {A_: set(), R_: set()}
    pred = {A_: set(), R_: set()}
    for d, ents in train_gold.items():
        for ent in ents:
            if len(ent) >= 3 and ent[2] in gold:
                gold[ent[2]].add((str(d), int(ent[0]), int(ent[1])))
    for d, text in train_texts.items():
        for s, e, lab in counsel_spans(text, policy):
            pred[lab].add((str(d), s, e))
    result = {}
    for lab in (A_, R_):
        tp = len(gold[lab] & pred[lab])
        p = tp / len(pred[lab]) if pred[lab] else 0.0
        r = tp / len(gold[lab]) if gold[lab] else 0.0
        result[lab] = 2*p*r/(p+r) if p+r else 0.0
    return result

_policy_scores = {
    p: _counsel_fold1_score(p) for p in ("R", "A", "drop")
}
chosen = max(
    _policy_scores,
    key=lambda p: (
        _policy_scores[p][A_] + _policy_scores[p][R_],
        p == "R"
    )
)
print("\nFold-1 counsel policy scores:")
for p, vals in _policy_scores.items():
    print(p, vals)
print("Selected counsel policy:", chosen)

# ------------------------------------------------------------
# 3. Original judge-pattern family and fold-1-only selection
# ------------------------------------------------------------
CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"
TOK_RE = re.compile(WORD)

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
B2_RE = re.compile(
    r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|"
    r"held\b|was\b|said\b|while\b|who\b|delivered|dissent|"
    r"agreed|also\b|noted|stated))"
)
C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE = re.compile(
    r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*"
    r"([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)"
    r"[ \t]*\(J\)"
)
E_RE = re.compile(
    r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]"
)
F_RE = re.compile(
    r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)"
    r"(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))"
)
JUDGE_NAMES = [
    "A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)",
    "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"
]
_STOP = {
    "JUDGMENT", "JUDGEMENT", "ORDER", "BENCH", "CORAM", "BEFORE",
    "PRESENT", "HON'BLE", "MR", "MRS", "MS", "JUSTICE", "THE",
    "AND", "OF", "DATE", "NEW", "DELHI", "BY", "DELIVERED"
}

def _trim_judge(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"):
        s += 1
    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"):
        e -= 1
    return (s, e) if e > s else None

def _back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [(m.start()+ws, m.end()+ws) for m in TOK_RE.finditer(text[ws:end])]
    run = []
    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t") or text[a:b].upper().strip(".") in _STOP:
            break
        run.insert(0, (a, b))
        if len(run) >= max_toks:
            break
    return (run[0][0], run[-1][1]) if run else None

def judge_patterns(text):
    out = {n: set() for n in JUDGE_NAMES}
    def add(n, s, e):
        span = _trim_judge(text, s, e)
        if span:
            out[n].add(span)
    for m in AJ_RE.finditer(text):
        add(JUDGE_NAMES[0], m.start(1), m.end(1))
    for rx, n in ((B_RE, JUDGE_NAMES[1]), (B2_RE, JUDGE_NAMES[2])):
        for m in rx.finditer(text):
            span = _back_name(text, m.start())
            if span:
                add(n, *span)
    for rx, n in ((C_RE, JUDGE_NAMES[3]), (E_RE, JUDGE_NAMES[5])):
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if inner and inner[0].isupper() and inner.upper() not in {"J", "JJ"} \
                    and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner):
                add(n, m.start(1), m.end(1))
    for m in D_RE.finditer(text):
        add(JUDGE_NAMES[4], m.start(1), m.end(1))
    for m in F_RE.finditer(text):
        seg, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", seg):
            k = seg.find(part, pos)
            if k < 0:
                continue
            pos = k + len(part)
            part2 = re.sub(r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part)
            span = _trim_judge(text, base+k, base+k+len(part2))
            if span and re.fullmatch(NAME2, text[span[0]:span[1]]):
                out[JUDGE_NAMES[6]].add(span)
    return out

def _resolve_judges(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1]-x[0]))):
        if out and s < out[-1][1]:
            continue
        out.append((s, e))
    return out

_gold_judge = {
    (str(d), int(ent[0]), int(ent[1]))
    for d, ents in train_gold.items()
    for ent in ents if len(ent) >= 3 and str(ent[2]) == "JUDGE"
}
_pattern_preds = {n: set() for n in JUDGE_NAMES}
for d, text in train_texts.items():
    for name, spans in judge_patterns(text).items():
        _pattern_preds[name].update((str(d), s, e) for s, e in spans)

keep = []
for name in JUDGE_NAMES:
    pred = _pattern_preds[name]
    tp = len(pred & _gold_judge)
    precision = tp / len(pred) if pred else 0.0
    if precision >= 0.40 and tp >= 5:
        keep.append(name)

print("\nFold-1 selected judge patterns:", keep)

def judge_spans_doc(text, selected):
    candidates = set()
    for name, spans in judge_patterns(text).items():
        if name in selected:
            candidates.update(spans)
    return _resolve_judges(candidates)

# ------------------------------------------------------------
# 4. Rebuild original merge behavior, evaluate baseline, then WIT
# ------------------------------------------------------------
def merge_all(texts, champ_filtered, selected_judges):
    result = {}
    for d, text in texts.items():
        rules = list(counsel_spans(text, chosen)) + [
            (s, e, "JUDGE")
            for s, e in judge_spans_doc(text, selected_judges)
        ]
        kept = [
            span for span in champ_filtered.get(d, [])
            if span[2] not in (A_, R_, "JUDGE")
        ]
        kept = [
            span for span in kept
            if not any(span[0] < re_ and span[1] > rs
                       for rs, re_, _ in rules)
        ]
        result[d] = sorted(set(kept + rules))
    return result

In [ ]:

# CS781 — Original honorific-based counsel rules + V5 side assignment
# Restored from cs781_champion_plus_rules_run_cells.ipynb
# Policy is selected on fold 1 only.

import re

HON_RE = re.compile(
    r"\b(?:Mr|Ms|Mrs|Shri|Sri|Smt)\b\.?[ \t]*(?=[A-Z])"
)
NAME_TOK = re.compile(r"[A-Z][A-Za-z\.']*")
WS = re.compile(r"[ \t]+")
SIDE_A = re.compile(
    r"petitioner|appellant|applicant|plaintiff|complainant|prosecution", re.I
)
SIDE_R = re.compile(r"respondent|defendant|accused", re.I)

def rule_spans(text):
    out = []
    for m in HON_RE.finditer(text):
        pos = m.end()
        if text[pos:pos + 7] == "Justice":
            continue
        p, name_end = pos, None
        while True:
            t = NAME_TOK.match(text, p)
            if not t:
                break
            name_end = t.end()
            w = WS.match(text, name_end)
            if not w or not text[w.end():w.end() + 1].isupper():
                break
            p = w.end()
        if name_end is None:
            continue
        e = name_end
        while e > pos and text[e - 1] in ".,;:'":
            e -= 1
        if e > pos:
            out.append((pos, e))
    return out

PARTY_A = (
    r"(?:petitioners?|appellants?|applicants?|plaintiffs?|"
    r"complainants?|prosecution|revisionists?)"
)
PARTY_R = (
    r"(?:respondents?|defendants?|accused|non-\s*applicants?|"
    r"opposite\s+part(?:y|ies))"
)
FOR = r"(?:for|on\s+behalf\s+of|appearing\s+for|representing)"
MODS = r"(?:(?:learned|first|second|third|\d+\w*|all|other)\s+)*"

CUE_A = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_A}", re.I)
CUE_R = re.compile(rf"\b{FOR}\s+(?:the\s+)?{MODS}{PARTY_R}", re.I)
HDR_A = re.compile(rf"{PARTY_A}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HDR_R = re.compile(rf"{PARTY_R}s?[\s.\-:]*(?:through|by)\s*[:\-]?\s*$", re.I)
HON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\.?[ \t]*$")
A_, R_ = "A.COUNSEL", "R.COUNSEL"
ORDER = ["hdr", "right", "left", "inherit"]

def _cues(text, lo, hi):
    out = [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]
    out += [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]
    return out

def assign_side(text, s, e, order, prev, W_L=300, W_R=200):
    for step in order:
        if step == "right":
            cues = _cues(text, e, min(len(text), e + W_R))
            if cues:
                return min(cues)[2], "right"
        elif step == "left":
            cues = _cues(text, max(0, s - W_L), s)
            cues = [x for x in cues if x[1] <= s]
            if cues:
                return max(cues, key=lambda x: x[1])[2], "left"
        elif step == "hdr":
            tail = HON_TAIL.sub("", text[max(0, s - 80):s])
            if HDR_A.search(tail):
                return A_, "hdr"
            if HDR_R.search(tail):
                return R_, "hdr"
        elif step == "inherit":
            if prev and s - prev[0] <= 200 and prev[2] != "default":
                return prev[1], "inherit"
    return R_, "default"

def counsel_spans(text, policy):
    out, prev = [], None
    for s, e in sorted(rule_spans(text)):
        lab, src = assign_side(text, s, e, ORDER, prev)
        prev = (e, lab, src)
        if src == "default":
            if policy == "drop":
                continue
            if policy == "A":
                lab = A_
        out.append((int(s), int(e), lab))
    return out

def counsel_f1(texts, gold, policy):
    G = {A_: set(), R_: set()}
    P = {A_: set(), R_: set()}
    for d, ents in gold.items():
        for ent in ents:
            if len(ent) >= 3 and ent[2] in G:
                G[ent[2]].add((str(d), int(ent[0]), int(ent[1])))
    for d, text in texts.items():
        for s, e, lab in counsel_spans(text, policy):
            P[lab].add((str(d), s, e))
    result = {}
    for lab in (A_, R_):
        tp = len(G[lab] & P[lab])
        p = tp / len(P[lab]) if P[lab] else 0.0
        r = tp / len(G[lab]) if G[lab] else 0.0
        result[lab] = 2*p*r/(p+r) if p+r else 0.0
    return result

fold1_policy_scores = {
    policy: counsel_f1(train_texts, train_gold, policy)
    for policy in ("R", "A", "drop")
}
chosen = max(
    fold1_policy_scores,
    key=lambda p: (
        fold1_policy_scores[p][A_] + fold1_policy_scores[p][R_],
        p == "R"
    )
)

print("Fold-1-only counsel policy selection:")
for policy, vals in fold1_policy_scores.items():
    print(
        f"{policy:5s} A={vals[A_]:.4f} R={vals[R_]:.4f} "
        f"mean={(vals[A_] + vals[R_])/2:.4f}"
    )
print("Selected policy:", chosen)
print("Counsel rule functions restored.")


In [ ]:

# CS781 — Original fold-1 judge-pattern selection + verified WIT hybrid
# No model training and no checkpoint/prediction files are written.

import re

CAP = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'’\-]*"
INIT = r"[A-Z]\.(?:[ \t]*[A-Z]\.)*"
WORD = rf"(?:{INIT}|{CAP})"
NAME2 = rf"{WORD}(?:[ \t]+{WORD}){{0,3}}"
TOK_RE = re.compile(WORD)

AJ_RE = re.compile(rf"(?i:\bjustice)[ \t]+({NAME2})")
B_RE = re.compile(r",[ \t]*J{1,2}\b\.?")
B2_RE = re.compile(
    r"[ \t]J{1,2}\.?(?=[ \t]*(?:in\b|speaking|writing|observed|"
    r"held\b|was\b|said\b|while\b|who\b|delivered|dissent|"
    r"agreed|also\b|noted|stated))"
)
C_RE = re.compile(r"\bJJ?\.?[ \t]*\(([^()\n]{3,45})\)")
D_RE = re.compile(
    r"(?:BENCH\s*:|\(J\)|\(J\.\))\s*"
    r"([A-Z][A-Za-z']+,[ \t]*[A-Z](?:[ \t]*\.?[ \t]*[A-Z])*\.?)"
    r"[ \t]*\(J\)"
)
E_RE = re.compile(
    r"\bJJ?\.?[ \t]*\[[ \t]*([A-Za-zÀ-ÿ\.'’ \t\-]{3,45}?)[ \t]*\]"
)
F_RE = re.compile(
    r"BENCH[ \t]*:[ \t]*([^\n]{3,120}?)"
    r"(?=[ \t]*(?:JUDG(?:E)?MENT|CITATION|ACT\b|\n|$))"
)
JUDGE_NAMES = [
    "A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)",
    "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"
]
STOP_WORDS = {
    "JUDGMENT", "JUDGEMENT", "ORDER", "BENCH", "CORAM", "BEFORE",
    "PRESENT", "HON'BLE", "MR", "MRS", "MS", "JUSTICE", "THE",
    "AND", "OF", "DATE", "NEW", "DELHI", "BY", "DELIVERED"
}

def trim(text, s, e):
    while s < e and (text[s].isspace() or text[s] in ".,;:"):
        s += 1
    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"):
        e -= 1
    return (s, e) if e > s else None

def back_name(text, end, max_toks=4):
    ws = max(0, end - 80)
    toks = [
        (m.start() + ws, m.end() + ws)
        for m in TOK_RE.finditer(text[ws:end])
    ]
    run = []
    for a, b in reversed(toks):
        nxt = run[0][0] if run else end
        if text[b:nxt].strip(" \t"):
            break
        if text[a:b].upper().strip(".") in STOP_WORDS:
            break
        run.insert(0, (a, b))
        if len(run) == max_toks:
            break
    return (run[0][0], run[-1][1]) if run else None

def judge_patterns2(text):
    out = {name: set() for name in JUDGE_NAMES}

    def add(name, s, e):
        span = trim(text, s, e)
        if span:
            out[name].add(span)

    for m in AJ_RE.finditer(text):
        add(JUDGE_NAMES[0], m.start(1), m.end(1))

    for rx, name in ((B_RE, JUDGE_NAMES[1]), (B2_RE, JUDGE_NAMES[2])):
        for m in rx.finditer(text):
            span = back_name(text, m.start())
            if span:
                add(name, *span)

    for rx, name in ((C_RE, JUDGE_NAMES[3]), (E_RE, JUDGE_NAMES[5])):
        for m in rx.finditer(text):
            inner = m.group(1).strip()
            if (
                inner and inner[0].isupper()
                and inner.upper() not in {"J", "JJ"}
                and re.fullmatch(r"[A-Za-zÀ-ÿ\.'’, \-]+", inner)
            ):
                add(name, m.start(1), m.end(1))

    for m in D_RE.finditer(text):
        add(JUDGE_NAMES[4], m.start(1), m.end(1))

    for m in F_RE.finditer(text):
        segment, base, pos = m.group(1), m.start(1), 0
        for part in re.split(r"[ \t]*(?:&|\bAND\b)[ \t]*", segment):
            k = segment.find(part, pos)
            if k < 0:
                continue
            pos = k + len(part)
            part2 = re.sub(
                r"[ \t]*\((?:C?J|J\.?)\)\s*$", "", part
            )
            span = trim(text, base + k, base + k + len(part2))
            if span and re.fullmatch(NAME2, text[span[0]:span[1]]):
                out[JUDGE_NAMES[6]].add(span)

    return out

def resolve_judge_overlaps(spans):
    out = []
    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1]-x[0]))):
        if out and s < out[-1][1]:
            continue
        out.append((s, e))
    return out

# Select patterns on fold 1 only.
gold_judge_train = {
    (str(d), int(ent[0]), int(ent[1]))
    for d, ents in train_gold.items()
    for ent in ents
    if len(ent) >= 3 and str(ent[2]) == "JUDGE"
}
pattern_preds = {name: set() for name in JUDGE_NAMES}

for d, text in train_texts.items():
    for name, spans in judge_patterns2(text).items():
        pattern_preds[name].update(
            (str(d), s, e) for s, e in spans
        )

keep = []
print("Fold-1-only judge-pattern selection:")
for name in JUDGE_NAMES:
    pred = pattern_preds[name]
    tp = len(gold_judge_train & pred)
    precision = tp / len(pred) if pred else 0.0
    recall = tp / len(gold_judge_train) if gold_judge_train else 0.0
    print(
        f"{name:26s} predicted={len(pred):4d} TP={tp:3d} "
        f"P={precision:.3f} R={recall:.3f}"
    )
    if precision >= 0.40 and tp >= 5:
        keep.append(name)

print("Selected judge patterns:", keep)

def judge_spans_doc(text, selected):
    spans = set()
    for name, candidates in judge_patterns2(text).items():
        if name in selected:
            spans.update(candidates)
    return resolve_judge_overlaps(spans)

# Use the established champion thresholds, not finalv1's altered champ_thr.
ESTABLISHED_THR = {
    "APP": 0.15,
    "RESP": 0.45,
    "A.COUNSEL": 0.45,
    "R.COUNSEL": 0.45,
    "JUDGE": 0.25,
    "WIT": 0.45,
    "AUTH": 0.35,
    "COURT": 0.45,
    "STAT": 0.25,
    "PREC": 0.35,
    "DATE": 0.55,
    "CASENO": 0.35,
}

# Verify the inference helpers and candidate rules exist.
for name in (
    "champion_model", "predict_texts", "apply_thr",
    "calculate_exact_span_metrics", "ENTITY_LABELS",
    "val_rule_candidates", "val_texts", "val_gold"
):
    if name not in globals():
        raise RuntimeError(f"Missing required object: {name}")

# Build baseline: thresholded champion + original counsel + selected judges.
raw_val_original = predict_texts(champion_model, val_texts)
filtered_val_original = apply_thr(raw_val_original, ESTABLISHED_THR)

def merge_all_original(texts, champion_filtered, selected_judges):
    result = {}
    for d, text in texts.items():
        rules = list(counsel_spans(text, chosen))
        rules += [
            (s, e, "JUDGE")
            for s, e in judge_spans_doc(text, selected_judges)
        ]

        kept = [
            span for span in champion_filtered.get(d, [])
            if span[2] not in ("A.COUNSEL", "R.COUNSEL", "JUDGE")
        ]
        kept = [
            span for span in kept
            if not any(
                span[0] < rule_e and span[1] > rule_s
                for rule_s, rule_e, _ in rules
            )
        ]
        result[d] = sorted(set(kept + rules))
    return result

In [ ]:
# Self-contained WIT helper definitions for the new experiments.
# No baseline globals or previous evaluation cells are required.

import re


_WIT_ID = re.compile(
    r"\b(?:P\s*\.?\s*W|D\s*\.?\s*W|C\s*\.?\s*W|L\s*\.?\s*W)"
    r"\s*[-.]?\s*\d+[A-Z]?\b", re.I
)

# Terms seen in the error audit that are not person-name spans in context.
_NON_PERSON = {
    "it", "both", "pw", "dw", "cw", "lw", "a", "an", "the",
    "dr", "court", "superintendent", "ex-sarpanch",
    "witness", "evidence", "statement", "respondent", "appellant",
    "accused", "petitioner", "deceased", "complainant",
}

def _near_identifier(text, s, e, radius=60):
    region = text[max(0, s-radius):min(len(text), e+radius)]
    return bool(_WIT_ID.search(region))

def _candidate_surface(text, s, e):
    return text[s:e].strip(" \t\r\n,.;:()[]{}-–—")

def _valid_person_surface(surface, mode):
    normalized = surface.casefold().strip()
    words = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", surface)

    if normalized in _NON_PERSON:
        return False

    # Reject fragments that are just witness-ID tokens.
    if re.fullmatch(r"(?:P|D|C|L)?W\d*", normalized.replace("-", "")):
        return False

    if mode in ("non_person", "name_shape", "name_shape_plus_context"):
        # Require at least one alphabetic word of reasonable length.
        if not words or max(map(len, words)) < 3:
            return False

    if mode in ("name_shape", "name_shape_plus_context"):
        # A single token can be a legitimate name, so do not reject it
        # merely for having one word.
        if len(surface.strip()) < 4:
            return False

    return True

In [ ]:
# NEW EXPERIMENTS — weighted cross-entropy vs focal loss
# IMPORTANT: run this notebook top-to-bottom; this cell no longer depends on old result variables.
# Both runs start independently from the champion checkpoint.
# Fold 1 trains; fold 2 selects/evaluates. Test labels are never used.
# Existing champion and predictions are not overwritten.

import copy, gc, json, math
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

SEED = 78115
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

VERIFIED_THR = {
    "APP": 0.15, "RESP": 0.45, "A.COUNSEL": 0.45, "R.COUNSEL": 0.45,
    "JUDGE": 0.25, "WIT": 0.45, "AUTH": 0.35, "COURT": 0.45,
    "STAT": 0.25, "PREC": 0.35, "DATE": 0.55, "CASENO": 0.35,
}
REFERENCE_SCORE = 0.333307
WIT_EXCLUSIONS = {
    "associate professor", "both pw", "manchili",
    "oblique", "subsequently", "the police patil",
}
TRIM_CHARS = " \t\r\n,.;:()[]{}"

# ---------- Build exactly the same validated hybrid pipeline ----------
def exp_full_pipeline(model, texts, gold=None):
    model.eval()
    raw = predict_texts(model, texts)
    thresholded = apply_thr(raw, VERIFIED_THR)
    base = merge_all_original(texts, thresholded, keep)

    result = {
        str(d): set((int(s), int(e), str(lab)) for s, e, lab in spans)
        for d, spans in base.items()
    }

    # Use the exact WIT candidate generator and validated filters.
    candidates_by_doc = (
        val_rule_candidates if texts is val_texts else
        {str(d): sorted(candidates_for_text(t)) for d, t in texts.items()}
    )
    for did, candidates in candidates_by_doc.items():
        did = str(did)
        if did not in texts:
            continue
        text = texts[did]
        pred = result.setdefault(did, set())

        for s, e, label in candidates:
            if str(label) != "WIT":
                continue
            s, e = int(s), int(e)
            surface = _candidate_surface(text, s, e)
            if not _near_identifier(text, s, e, radius=60):
                continue
            if not _valid_person_surface(surface, "name_shape_plus_context"):
                continue
            if not _near_identifier(text, s, e, radius=30):
                continue
            if surface.casefold() in WIT_EXCLUSIONS:
                continue
            if any(max(s, ps) < min(e, pe) for ps, pe, _ in pred):
                continue
            pred.add((s, e, "WIT"))

    # Best validated boundary configuration: trim CASENO + STAT only.
    final = {}
    for did, spans in result.items():
        text = texts[did]
        cleaned = set()
        for s, e, lab in spans:
            if lab in ("CASENO", "STAT"):
                while s < e and text[s] in TRIM_CHARS:
                    s += 1
                while e > s and text[e-1] in TRIM_CHARS:
                    e -= 1
            if s < e:
                cleaned.add((int(s), int(e), str(lab)))
        final[did] = sorted(cleaned)

    if gold is None:
        return final
    return final, calculate_exact_span_metrics(gold, final, ENTITY_LABELS)

print("=== Rebuild reference hybrid before training ===")
champion_model.eval()
reference_predictions, reference_metrics = exp_full_pipeline(
    champion_model, val_texts, val_gold
)
reference_score = float(reference_metrics["MACRO_F1"])
print(f"Rebuilt score: {reference_score:.6f} | target: {REFERENCE_SCORE:.6f}")
if abs(reference_score - REFERENCE_SCORE) > 0.002:
    raise RuntimeError(
        "Reference pipeline mismatch; no experiment was trained. "
        "The rebuilt pipeline must match the known 0.333307 baseline."
    )

# ---------- Prepare fold-1 BIO training chunks ----------
MAX_LEN, STRIDE, BATCH_SIZE = 512, 448, 1
EPOCHS = 1
ENCODER_LR = 5e-6
CLASSIFIER_LR = 2e-5

def encode_train_doc(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = list(enc.ids), list(enc.offsets)
    tags = [label2id["O"]] * len(ids)
    conflicts = [False] * len(ids)

    for ent in entities:
        if len(ent) < 3:
            continue
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        if lab not in ENTITY_LABELS or s < 0 or e <= s or e > len(text):
            continue
        tok_ix = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]
        if not tok_ix:
            continue
        for j, i in enumerate(tok_ix):
            tag_id = label2id[("B-" if j == 0 else "I-") + lab]
            if tags[i] != label2id["O"] and tags[i] != tag_id:
                conflicts[i] = True
            else:
                tags[i] = tag_id

    for i, bad in enumerate(conflicts):
        if bad:
            tags[i] = -100

    chunks, seen = [], set()
    for start in range(0, len(ids), STRIDE):
        end = min(start + MAX_LEN, len(ids))
        if start >= end:
            break
        chunk_labels = []
        for i in range(start, end):
            if i in seen:
                chunk_labels.append(-100)
            else:
                chunk_labels.append(tags[i])
                seen.add(i)
        chunks.append({"input_ids": ids[start:end], "labels": chunk_labels})
        if end == len(ids):
            break
    return chunks, tags

train_chunks = []
tag_counts = torch.zeros(len(BIO_LABELS), dtype=torch.long)
for did, text in train_texts.items():
    chunks, full_tags = encode_train_doc(text, train_gold[did])
    train_chunks.extend(chunks)
    for tag in full_tags:
        if tag >= 0:
            tag_counts[tag] += 1

assert train_chunks
print("Training docs:", len(train_texts), "| chunks:", len(train_chunks))
print("Validation docs:", len(val_texts), "| test docs:", len(test_texts))

def collate_train(batch):
    n = max(len(x["input_ids"]) for x in batch)
    ids = torch.zeros((len(batch), n), dtype=torch.long)
    mask = torch.zeros_like(ids)
    labels = torch.full((len(batch), n), -100, dtype=torch.long)
    for i, item in enumerate(batch):
        k = len(item["input_ids"])
        ids[i, :k] = torch.tensor(item["input_ids"], dtype=torch.long)
        mask[i, :k] = 1
        labels[i, :k] = torch.tensor(item["labels"], dtype=torch.long)
    return ids, mask, labels

train_loader = DataLoader(
    train_chunks, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_train
)

# New experiment A: bounded inverse-frequency weighted cross entropy.
weights = tag_counts.float().clamp_min(1).pow(-0.25)
weights = (weights / weights.mean()).clamp(0.5, 3.0).to(device)

# New experiment B: focal loss (gamma=2) with the same bounded class alpha.
def focal_token_loss(logits, labels, alpha, gamma=2.0):
    flat_logits = logits.reshape(-1, logits.shape[-1])
    flat_labels = labels.reshape(-1)
    valid = flat_labels != -100
    if not valid.any():
        return flat_logits.sum() * 0.0
    logits = flat_logits[valid]
    labels = flat_labels[valid]
    log_probs = torch.log_softmax(logits, dim=-1)
    log_pt = log_probs.gather(1, labels[:, None]).squeeze(1)
    pt = log_pt.exp()
    a = alpha[labels]
    return (-a * ((1.0 - pt).clamp_min(0) ** gamma) * log_pt).mean()

def run_training_variant(name, loss_kind):
    # Independent model copy from the same untouched champion.
    model = copy.deepcopy(champion_model).to(device)
    model.train()
    optimizer = torch.optim.AdamW(
        [
            {"params": model.encoder.parameters(), "lr": ENCODER_LR},
            {"params": model.classifier.parameters(), "lr": CLASSIFIER_LR},
        ],
        weight_decay=0.01,
    )
    ce_loss = nn.CrossEntropyLoss(weight=weights, ignore_index=-100)
    losses = []

    for epoch in range(EPOCHS):
        model.train()
        for ids, mask, labels in tqdm(
            train_loader, desc=f"{name} epoch {epoch+1}/{EPOCHS}"
        ):
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(ids, mask)
            if loss_kind == "weighted_ce":
                loss = ce_loss(
                    logits.reshape(-1, len(BIO_LABELS)), labels.reshape(-1)
                )
            elif loss_kind == "focal":
                loss = focal_token_loss(logits, labels, weights, gamma=2.0)
            else:
                raise ValueError(loss_kind)

            if not torch.isfinite(loss):
                raise RuntimeError(f"{name}: non-finite loss.")
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(float(loss.item()))

    model.eval()
    predictions, metrics = exp_full_pipeline(model, val_texts, val_gold)
    score = float(metrics["MACRO_F1"])
    print(f"\n{name}: mean loss={np.mean(losses):.5f}, macro-F1={score:.6f}, "
          f"delta={score-reference_score:+.6f}")
    for lab in ENTITY_LABELS:
        old, new = reference_metrics[lab]["f1"], metrics[lab]["f1"]
        print(f"  {lab:12s} {old:.4f} -> {new:.4f} ({new-old:+.4f})")

    return {
        "name": name,
        "score": score,
        "metrics": metrics,
        "predictions": predictions,
        "model": model,
    }

# Run two distinct, one-epoch experiments from the exact same starting checkpoint.
expA = run_training_variant("Experiment A — weighted cross-entropy", "weighted_ce")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

expB = run_training_variant("Experiment B — focal loss (gamma=2)", "focal")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

results = sorted([expA, expB], key=lambda x: x["score"], reverse=True)
print("\n=== EXPERIMENT LEADERBOARD ===")
print(f"Reference hybrid: {reference_score:.6f}")
for r in results:
    print(f"{r['name']}: {r['score']:.6f} (delta {r['score']-reference_score:+.6f})")

# Save each experimental checkpoint only if it beats the known reference.
for r in results:
    if r["score"] > REFERENCE_SCORE:
        out = Path("/kaggle/working") / (
            "experiment15_weighted_ce.pt"
            if r["name"].endswith("weighted cross-entropy")
            else "experiment16_focal_loss.pt"
        )
        torch.save({
            "model_state_dict": r["model"].state_dict(),
            "bio_labels": BIO_LABELS,
            "experiment": r["name"],
            "validation_macro_f1": float(r["score"]),
            "reference_macro_f1": float(REFERENCE_SCORE),
            "epochs": EPOCHS,
        }, out)
        print("Saved improved candidate checkpoint:", out)

if all(r["score"] <= REFERENCE_SCORE for r in results):
    print("Neither new experiment beat the reference; champion remains unchanged.")

print("\nNo test gold labels were used.")
print("Champion weights and existing prediction files were not modified.")

In [ ]:

import random
import re
from collections import defaultdict, Counter

# Use a fixed seed so the generated examples are reproducible.
AUG_SEED = 781
_rng = random.Random(AUG_SEED)

# Expected formats:
# train_texts: {doc_id: full_document_text}
# train_gold:  {doc_id: [[start, end, label], ...]}
#
# This cell assumes character offsets use [start, end).

def _normalise_span(span):
    return int(span[0]), int(span[1]), str(span[2])

replacement_pool = defaultdict(set)
entity_counts = Counter()
span_errors = []

for doc_id, text in train_texts.items():
    annotations = train_gold[str(doc_id)] if str(doc_id) in train_gold else train_gold[doc_id]

    for raw_span in annotations:
        start, end, label = _normalise_span(raw_span)

        if not (0 <= start < end <= len(text)):
            span_errors.append((doc_id, start, end, label, "out_of_bounds"))
            continue

        surface = text[start:end]
        if not surface.strip():
            span_errors.append((doc_id, start, end, label, "empty_surface"))
            continue

        # Keep the exact surface, including punctuation/casing.
        replacement_pool[label].add(surface)
        entity_counts[label] += 1

if span_errors:
    print("Invalid training spans:", len(span_errors))
    print("First few:", span_errors[:10])
    raise ValueError("Fix invalid training annotations before augmentation.")

print("Training documents:", len(train_texts))
print("Training entities:", sum(entity_counts.values()))
print("Replacement candidates by label:")
for label in sorted(replacement_pool):
    print(f"{label:12s} {len(replacement_pool[label]):5d} unique surfaces")


In [ ]:

def make_entity_replacement_example(text, annotations, rng, max_replacements=2):
    """
    Replace a small number of annotated mentions with same-label
    surfaces from the training-only replacement pools.

    Returns (new_text, new_annotations).
    Annotations use [start, end, label], with end exclusive.
    """
    spans = sorted(
        [_normalise_span(s) for s in annotations],
        key=lambda x: (x[0], x[1])
    )

    # Verify bounds and non-overlap before modifying anything.
    previous_end = 0
    for start, end, label in spans:
        if not (0 <= start < end <= len(text)):
            raise ValueError(f"Invalid span: {(start, end, label)}")
        if start < previous_end:
            raise ValueError("Overlapping gold spans detected.")
        previous_end = end

    eligible = []
    for i, (start, end, label) in enumerate(spans):
        original_surface = text[start:end]
        options = [
            x for x in replacement_pool.get(label, set())
            if x != original_surface
            and x.strip()
            and "\n" not in x
            and "\r" not in x
        ]
        if options:
            eligible.append((i, options))

    if not eligible:
        return text, spans

    rng.shuffle(eligible)
    selected = {
        i: rng.choice(options)
        for i, options in eligible[:max_replacements]
    }

    pieces = []
    new_annotations = []
    cursor = 0
    new_length = 0

    for i, (start, end, label) in enumerate(spans):
        prefix = text[cursor:start]
        pieces.append(prefix)
        new_length += len(prefix)

        surface = selected.get(i, text[start:end])
        new_start = new_length
        pieces.append(surface)
        new_length += len(surface)
        new_end = new_length

        new_annotations.append([new_start, new_end, label])
        cursor = end

    pieces.append(text[cursor:])
    new_text = "".join(pieces)

    # Mandatory exact-offset integrity check.
    for start, end, label in new_annotations:
        assert 0 <= start < end <= len(new_text)
        assert new_text[start:end].strip()

    return new_text, new_annotations


# Smoke test on a single training document.
_smoke_id = next(iter(train_texts))
_smoke_text = train_texts[_smoke_id]
_smoke_gold = train_gold[str(_smoke_id)] if str(_smoke_id) in train_gold else train_gold[_smoke_id]

_smoke_aug_text, _smoke_aug_gold = make_entity_replacement_example(
    _smoke_text, _smoke_gold, random.Random(AUG_SEED)
)

assert len(_smoke_aug_gold) == len(_smoke_gold)
print("Smoke test passed.")
print("Original characters:", len(_smoke_text))
print("Augmented characters:", len(_smoke_aug_text))
print("Entity count preserved:", len(_smoke_aug_gold))


In [ ]:

aug_texts = dict(train_texts)
aug_gold = {
    doc_id: [
        list(_normalise_span(s))
        for s in (train_gold[str(doc_id)] if str(doc_id) in train_gold else train_gold[doc_id])
    ]
    for doc_id in train_texts
}

# One synthetic variant per original training document.
for doc_id, text in train_texts.items():
    gold = aug_gold[doc_id]
    new_text, new_gold = make_entity_replacement_example(
        text, gold, random.Random(f"{AUG_SEED}-{doc_id}"),
        max_replacements=2
    )

    # Add only examples that actually changed.
    if new_text != text:
        synthetic_id = f"{doc_id}__entity_replacement"
        aug_texts[synthetic_id] = new_text
        aug_gold[synthetic_id] = new_gold

assert set(train_texts).isdisjoint(
    set(aug_texts) - set(train_texts)
)

print("Original training documents:", len(train_texts))
print("Synthetic documents added:", len(aug_texts) - len(train_texts))
print("Total training documents:", len(aug_texts))


In [ ]:

# This cell intentionally does not train on any additional data.
# It verifies that the named training/validation/test pools are separate.

train_ids = set(map(str, train_texts.keys()))
val_ids = set(map(str, val_texts.keys()))

test_ids = set(map(str, test_texts.keys())) if "test_texts" in globals() else set()

assert train_ids.isdisjoint(val_ids), "Training/validation ID overlap!"
assert train_ids.isdisjoint(test_ids), "Training/test ID overlap!"
assert val_ids.isdisjoint(test_ids), "Validation/test ID overlap!"

print("Training documents:", len(train_ids))
print("Validation documents:", len(val_ids))
print("Test documents:", len(test_ids))
print()
print("No separate eligible unlabeled pool is declared by this cell.")
print("Only proceed with pseudo-labeling if you identify another official")
print("Phase 2 training-text pool and confirm its use is permitted.")


In [ ]:

from collections import Counter

def validate_character_annotations(texts, annotations_by_doc):
    errors = []
    checked_spans = 0

    for doc_id, text in texts.items():
        spans = annotations_by_doc.get(doc_id)

        if spans is None:
            errors.append((doc_id, "missing_annotations"))
            continue

        ordered = sorted(
            [(int(s), int(e), str(label)) for s, e, label in spans],
            key=lambda x: (x[0], x[1])
        )

        previous_end = 0
        seen = set()

        for start, end, label in ordered:
            checked_spans += 1

            if not (0 <= start < end <= len(text)):
                errors.append((doc_id, "out_of_bounds", start, end, label))
                continue

            if start < previous_end:
                errors.append((doc_id, "overlapping_spans", start, end, label))

            if (start, end, label) in seen:
                errors.append((doc_id, "duplicate_span", start, end, label))

            if not text[start:end].strip():
                errors.append((doc_id, "empty_surface", start, end, label))

            seen.add((start, end, label))
            previous_end = max(previous_end, end)

    print("Documents checked:", len(texts))
    print("Spans checked:", checked_spans)
    print("Errors:", len(errors))

    if errors:
        print("First 20 errors:")
        for err in errors[:20]:
            print(err)
        raise ValueError("Character annotation validation failed.")

    print("PASS: all character annotations are valid.")


validate_character_annotations(aug_texts, aug_gold)


In [ ]:

def span_triplet(span):
    """Normalize an annotation to (start, end, label), ignoring extra fields."""
    if len(span) < 3:
        raise ValueError(f"Annotation has fewer than 3 fields: {span!r}")
    return [int(span[0]), int(span[1]), str(span[2])]


# Verify that original examples and their labels remain unchanged.
for doc_id, original_text in train_texts.items():
    assert aug_texts[doc_id] == original_text, (
        f"Original text changed for {doc_id}"
    )

    original_gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold
        else train_gold[doc_id]
    )

    expected = [span_triplet(s) for s in original_gold]
    actual = [span_triplet(s) for s in aug_gold[doc_id]]

    assert actual == expected, (
        f"Original annotations changed for {doc_id}"
    )

synthetic_ids = set(aug_texts) - set(train_texts)

print("Original training examples preserved.")
print("Synthetic examples:", len(synthetic_ids))
print(
    "Original documents with an augmented variant:",
    sum(aug_texts[d] != train_texts[d] for d in train_texts)
)


In [ ]:

shown = 0

for doc_id in sorted(synthetic_ids):
    original_id = doc_id.removesuffix("__entity_replacement")

    if original_id not in train_texts:
        continue

    original_text = train_texts[original_id]
    augmented_text = aug_texts[doc_id]

    print("=" * 80)
    print("Document:", original_id)
    print("Original excerpt:")
    print(original_text[:500])
    print("\nAugmented excerpt:")
    print(augmented_text[:500])

    shown += 1
    if shown >= 3:
        break

if shown == 0:
    print("No synthetic examples were generated.")


In [ ]:

# Diagnose why the generated variants did not change.

for doc_id in list(train_texts)[:5]:
    text = train_texts[doc_id]
    gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold
        else train_gold[doc_id]
    )

    eligible_count = 0
    for span in gold:
        start, end, label = int(span[0]), int(span[1]), str(span[2])
        surface = text[start:end]
        options = [
            x for x in replacement_pool.get(label, set())
            if x != surface
            and x.strip()
            and "\n" not in x
            and "\r" not in x
        ]
        if options:
            eligible_count += 1

    test_text, test_gold = make_entity_replacement_example(
        text, gold, random.Random(f"diagnostic-{doc_id}"),
        max_replacements=2
    )

    print(
        doc_id,
        "| eligible spans:", eligible_count,
        "| changed:", test_text != text,
        "| original chars:", len(text),
        "| new chars:", len(test_text)
    )


In [ ]:

# Rebuild synthetic variants from the original training data.
# Preserve original examples and annotations unchanged.

aug_texts = dict(train_texts)
aug_gold = {}

for doc_id, text in train_texts.items():
    gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold
        else train_gold[doc_id]
    )

    aug_gold[doc_id] = [
        [int(s[0]), int(s[1]), str(s[2])]
        for s in gold
    ]

changed_count = 0
unchanged_count = 0

for doc_id, text in train_texts.items():
    gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold
        else train_gold[doc_id]
    )

    new_text, new_gold = make_entity_replacement_example(
        text,
        gold,
        random.Random(f"{AUG_SEED}-{doc_id}"),
        max_replacements=2
    )

    synthetic_id = f"{doc_id}__entity_replacement"

    if new_text != text:
        aug_texts[synthetic_id] = new_text
        aug_gold[synthetic_id] = new_gold
        changed_count += 1
    else:
        unchanged_count += 1

assert len(aug_texts) == len(train_texts) + changed_count
assert all(
    aug_texts[d] == train_texts[d]
    for d in train_texts
)

validate_character_annotations(aug_texts, aug_gold)

print("Original training documents:", len(train_texts))
print("Synthetic variants created:", changed_count)
print("Documents with no eligible replacement:", unchanged_count)
print("Total training examples:", len(aug_texts))


In [ ]:

import re
from collections import Counter, defaultdict

def surface_shape(text):
    text = text.strip()
    if not text:
        return "empty"
    if re.fullmatch(r"[\d\s./()-]+", text):
        return "numeric_or_case_id"
    if re.fullmatch(r"[A-Z][A-Z .'-]*", text):
        return "uppercase_name_or_heading"
    if re.search(r"\d", text):
        return "contains_digits"
    if text[0].isupper():
        return "capitalized"
    return "other"

# Match each synthetic span against the corresponding original span
# by annotation order. This is a review aid, not a semantic proof.
replacement_audit = []

for doc_id in train_texts:
    synthetic_id = f"{doc_id}__entity_replacement"
    if synthetic_id not in aug_texts:
        continue

    original_text = train_texts[doc_id]
    synthetic_text = aug_texts[synthetic_id]

    original_gold = (
        train_gold[str(doc_id)]
        if str(doc_id) in train_gold else train_gold[doc_id]
    )
    synthetic_gold = aug_gold[synthetic_id]

    if len(original_gold) != len(synthetic_gold):
        raise ValueError(f"Entity count changed: {doc_id}")

    for old, new in zip(original_gold, synthetic_gold):
        old_start, old_end, old_label = int(old[0]), int(old[1]), str(old[2])
        new_start, new_end, new_label = int(new[0]), int(new[1]), str(new[2])

        old_surface = original_text[old_start:old_end]
        new_surface = synthetic_text[new_start:new_end]

        if old_label != new_label:
            raise ValueError(f"Label changed for {doc_id}")

        if old_surface != new_surface:
            replacement_audit.append({
                "doc_id": doc_id,
                "label": old_label,
                "old": old_surface,
                "new": new_surface,
                "old_shape": surface_shape(old_surface),
                "new_shape": surface_shape(new_surface),
            })

print("Actual replacements:", len(replacement_audit))
print("Replacements by label:")
for label, count in sorted(Counter(x["label"] for x in replacement_audit).items()):
    print(f"{label:12s} {count:4d}")

print("\nFirst 30 replacements to review:")
for item in replacement_audit[:30]:
    print(
        f'{item["label"]:12s} | {item["old"]!r} -> {item["new"]!r} '
        f'| {item["old_shape"]} -> {item["new_shape"]}'
    )


In [ ]:

import inspect

keywords = (
    "token", "bio", "align", "encode", "chunk",
    "label", "offset", "dataset", "collat"
)

matches = []
for name, obj in list(globals().items()):
    if not callable(obj):
        continue
    if not any(k in name.lower() for k in keywords):
        continue
    try:
        signature = str(inspect.signature(obj))
    except Exception:
        signature = "(signature unavailable)"
    matches.append((name, signature))

print("Relevant functions:")
for name, signature in sorted(matches):
    print(f"{name}{signature}")


In [ ]:

if "tokenizer" not in globals():
    print("No `tokenizer` variable found; use the tokenizer identified in Cell 2.")
else:
    sample_id = next(iter(aug_texts))
    sample_text = aug_texts[sample_id]
    sample_encoding = tokenizer.encode(sample_text)

    ids = sample_encoding.ids
    offsets = sample_encoding.offsets

    assert len(ids) == len(offsets), "Token IDs and offsets differ in length."

    nonempty_offsets = [
        (s, e) for s, e in offsets
        if isinstance(s, int) and isinstance(e, int) and e > s
    ]

    print("Sample:", sample_id)
    print("Text characters:", len(sample_text))
    print("Tokens:", len(ids))
    print("Tokens with non-empty offsets:", len(nonempty_offsets))
    print("First 20 offsets:", offsets[:20])


In [ ]:

import inspect
import traceback

print("encode_train_doc signature:")
print(inspect.signature(encode_train_doc))
print("\nFunction source:")
print(inspect.getsource(encode_train_doc))


In [ ]:

import re
from collections import Counter

def looks_like_citation(s):
    s = s.strip()
    return bool(
        re.search(r"\bv\.?\s", s, re.I)
        or re.search(r"\b(SCC|AIR|SCR|MANU)\b", s)
        or re.search(r"\b\d{4}\b", s)
    )

def looks_like_person(s):
    words = re.findall(r"[A-Za-z][A-Za-z.'’-]*", s)
    return (
        1 <= len(words) <= 6
        and all(w[0].isupper() for w in words)
        and not looks_like_citation(s)
    )

def suspicious_replacement(label, old, new):
    label = label.upper()
    old_citation = looks_like_citation(old)
    new_citation = looks_like_citation(new)

    if label == "PREC":
        return old_citation != new_citation
    if label == "DATE":
        return not bool(re.search(
            r"\d|January|February|March|April|May|June|July|August|"
            r"September|October|November|December",
            new, re.I
        ))
    if label == "CASENO":
        return not bool(re.search(r"\d", new))
    if label in {"APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT"}:
        return not looks_like_person(new)
    if label == "STAT":
        return not bool(re.search(
            r"\b(Act|Code|Rules|Constitution|Regulation|Ordinance)\b",
            new, re.I
        ))
    return False

flagged = [
    x for x in replacement_audit
    if suspicious_replacement(x["label"], x["old"], x["new"])
]

print("Replacements flagged for review:", len(flagged), "/", len(replacement_audit))
print("\nFlagged samples:")
for x in flagged[:60]:
    print(f'{x["label"]:12s} | {x["old"]!r} -> {x["new"]!r}')


In [ ]:

# Corrected token/BIO integrity audit.
# Overlapping chunks are expected; validate each chunk independently.

valid_ids = set(label2id.values())
errors = []
total_tokens = 0
total_entities = 0
ignored_tokens = 0
chunk_count = 0

for doc_id, text in aug_texts.items():
    entities = aug_gold[doc_id]
    chunks, tags = encode_train_doc(text, entities)

    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = list(enc.ids), list(enc.offsets)

    if len(ids) != len(tags):
        errors.append((doc_id, "token/tag length mismatch"))
        continue

    if any(t != -100 and t not in valid_ids for t in tags):
        errors.append((doc_id, "invalid BIO label ID"))

    # Validate each chunk against its corresponding original token window.
    expected_start = 0
    for chunk in chunks:
        chunk_count += 1
        chunk_ids = chunk["input_ids"]
        chunk_labels = chunk["labels"]

        if len(chunk_ids) != len(chunk_labels):
            errors.append((doc_id, "chunk input/label length mismatch"))
            continue

        if len(chunk_ids) > MAX_LEN:
            errors.append((doc_id, "chunk exceeds MAX_LEN"))

        if not chunk_ids:
            errors.append((doc_id, "empty chunk"))
            continue

        # Locate this chunk's token window in the original sequence.
        # The encoder uses starts at multiples of STRIDE.
        start = expected_start
        end = start + len(chunk_ids)

        if ids[start:end] != chunk_ids:
            errors.append((doc_id, "chunk tokens do not match expected window"))
            continue

        if end > len(ids):
            errors.append((doc_id, "chunk extends past token sequence"))
            continue

        expected_start += STRIDE
        if end == len(ids):
            expected_start = len(ids)

    # Verify the chunks cover the complete token sequence.
    if ids and not chunks:
        errors.append((doc_id, "no chunks returned"))
    elif chunks and len(ids) > MAX_LEN:
        last_start = (len(chunks) - 1) * STRIDE
        if last_start + len(chunks[-1]["input_ids"]) < len(ids):
            errors.append((doc_id, "last chunk does not cover document end"))

    # Every valid entity must overlap at least one token.
    for ent in entities:
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])

        if lab not in ENTITY_LABELS or s < 0 or e <= s or e > len(text):
            errors.append((doc_id, f"invalid entity span: {ent[:3]}"))
            continue

        overlap = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]
        if not overlap:
            errors.append((doc_id, f"entity has no token overlap: {ent[:3]}"))

        total_entities += 1

    total_tokens += len(ids)
    ignored_tokens += sum(t == -100 for t in tags)

print("Documents checked:", len(aug_texts))
print("Chunks checked:", chunk_count)
print("Tokens checked:", total_tokens)
print("Entities checked:", total_entities)
print("Ignored/conflict tokens:", ignored_tokens)
print("Errors:", len(errors))

for err in errors[:30]:
    print("ERROR:", err)

if not errors:
    print("PASS: token/BIO structural checks passed.")
else:
    print("FAIL: inspect the reported chunk-window errors before training.")


In [ ]:

# Final pre-training audit; does not train or overwrite model artifacts.

assert all(aug_texts[d] == train_texts[d] for d in train_texts)
assert all(d in aug_gold for d in train_texts)

synthetic_ids = [d for d in aug_texts if str(d).endswith("__entity_replacement")]

assert len(aug_texts) == len(train_texts) + len(synthetic_ids)

bad = []
for doc_id in synthetic_ids:
    text = aug_texts[doc_id]
    chunks, tags = encode_train_doc(text, aug_gold[doc_id])

    if not chunks or not tags:
        bad.append((doc_id, "empty encoding"))
        continue

    if len(tags) != len(tokenizer.encode(text, add_special_tokens=False).ids):
        bad.append((doc_id, "token/tag length mismatch"))

print("Original examples preserved:", len(train_texts))
print("Synthetic examples:", len(synthetic_ids))
print("Total examples:", len(aug_texts))
print("Encoding errors:", len(bad))

for item in bad[:20]:
    print("ERROR:", item)

assert not bad, "Resolve encoding errors before training."
print("PASS: final structural pre-training audit.")

In [ ]:

# EXPERIMENT: Entity-replacement augmentation
# Starts from champion weights. Never modifies champion_model or existing predictions.

import copy
import gc
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# ---------- Preconditions ----------
required = [
    "champion_model", "train_texts", "train_gold",
    "val_texts", "val_gold", "aug_texts", "aug_gold",
    "tokenizer", "BIO_LABELS", "label2id", "ENTITY_LABELS",
    "exp_full_pipeline",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing notebook variables: {missing}. Run the preceding setup cells first."
    )

synthetic_ids = [
    d for d in aug_texts
    if str(d).endswith("__entity_replacement")
]
assert len(train_texts) == 35, f"Expected 35 original training docs, got {len(train_texts)}"
assert len(synthetic_ids) > 0, "No synthetic variants found."
assert all(aug_texts[d] == train_texts[d] for d in train_texts)
assert set(train_texts).isdisjoint(set(val_texts))
assert set(val_texts) == set(val_gold)

# Fixed experiment settings taken from the existing notebook.
SEED_AUG_EXP = 78121
AUG_MAX_LEN = 512
AUG_STRIDE = 448
AUG_BATCH_SIZE = 1
AUG_EPOCHS = 1
AUG_ENCODER_LR = 5e-6
AUG_CLASSIFIER_LR = 2e-5

random.seed(SEED_AUG_EXP)
np.random.seed(SEED_AUG_EXP)
torch.manual_seed(SEED_AUG_EXP)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_AUG_EXP)

# Reconfirm the reference pipeline before training.
champion_model.eval()
_, aug_reference_metrics = exp_full_pipeline(
    champion_model, val_texts, val_gold
)
aug_reference_score = float(aug_reference_metrics["MACRO_F1"])

print(f"Champion hybrid validation macro-F1: {aug_reference_score:.6f}")
print("Expected reference: approximately 0.333307")

if abs(aug_reference_score - 0.333307) > 0.002:
    raise RuntimeError(
        "Reference score mismatch. Stop here and investigate before training."
    )

# ---------- Encode original + synthetic examples ----------
def encode_augmented_document(text, entities):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids, offsets = list(enc.ids), list(enc.offsets)
    tags = [label2id["O"]] * len(ids)
    conflicts = [False] * len(ids)

    for ent in entities:
        if len(ent) < 3:
            continue
        s, e, lab = int(ent[0]), int(ent[1]), str(ent[2])
        if lab not in ENTITY_LABELS or s < 0 or e <= s or e > len(text):
            raise ValueError(f"Invalid entity annotation: {ent[:3]}")

        tok_ix = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < e and b > s
        ]
        if not tok_ix:
            raise ValueError(f"Entity has no token overlap: {ent[:3]}")

        for j, i in enumerate(tok_ix):
            tag_id = label2id[("B-" if j == 0 else "I-") + lab]
            if tags[i] != label2id["O"] and tags[i] != tag_id:
                conflicts[i] = True
            else:
                tags[i] = tag_id

    for i, conflict in enumerate(conflicts):
        if conflict:
            tags[i] = -100

    chunks, seen = [], set()
    for start in range(0, len(ids), AUG_STRIDE):
        end = min(start + AUG_MAX_LEN, len(ids))
        if start >= end:
            break

        chunk_labels = []
        for i in range(start, end):
            if i in seen:
                chunk_labels.append(-100)
            else:
                chunk_labels.append(tags[i])
                seen.add(i)

        chunks.append({
            "input_ids": ids[start:end],
            "labels": chunk_labels,
        })
        if end == len(ids):
            break

    return chunks

aug_train_chunks = []
for doc_id, text in aug_texts.items():
    if doc_id not in aug_gold:
        raise KeyError(f"Missing annotations for {doc_id}")
    aug_train_chunks.extend(
        encode_augmented_document(text, aug_gold[doc_id])
    )

assert aug_train_chunks, "No training chunks were produced."

def collate_augmented(batch):
    max_len = max(len(item["input_ids"]) for item in batch)
    ids = torch.zeros((len(batch), max_len), dtype=torch.long)
    mask = torch.zeros_like(ids)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)

    for i, item in enumerate(batch):
        n = len(item["input_ids"])
        ids[i, :n] = torch.tensor(item["input_ids"], dtype=torch.long)
        mask[i, :n] = 1
        labels[i, :n] = torch.tensor(item["labels"], dtype=torch.long)

    return ids, mask, labels

generator = torch.Generator()
generator.manual_seed(SEED_AUG_EXP)

aug_loader = DataLoader(
    aug_train_chunks,
    batch_size=AUG_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_augmented,
    generator=generator,
)

print("Original documents:", len(train_texts))
print("Synthetic variants:", len(synthetic_ids))
print("Total examples:", len(aug_texts))
print("Training chunks:", len(aug_train_chunks))
print("Validation documents:", len(val_texts))
print("Test data will not be evaluated.")


In [ ]:

# Train a separate experimental model from the untouched champion.
# This cell never assigns to champion_model.

aug_model = copy.deepcopy(champion_model).to(device)
aug_model.train()

optimizer = torch.optim.AdamW(
    [
        {"params": aug_model.encoder.parameters(), "lr": AUG_ENCODER_LR},
        {"params": aug_model.classifier.parameters(), "lr": AUG_CLASSIFIER_LR},
    ],
    weight_decay=0.01,
)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

loss_history = []

for epoch in range(AUG_EPOCHS):
    aug_model.train()
    running_loss = 0.0
    batches = 0

    for ids, mask, labels in tqdm(
        aug_loader,
        desc=f"Entity-replacement experiment: epoch {epoch + 1}/{AUG_EPOCHS}",
    ):
        ids = ids.to(device)
        mask = mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = aug_model(ids, mask)

        loss = criterion(
            logits.reshape(-1, len(BIO_LABELS)),
            labels.reshape(-1),
        )

        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss; stopping experiment.")

        loss.backward()
        torch.nn.utils.clip_grad_norm_(aug_model.parameters(), 1.0)
        optimizer.step()

        running_loss += float(loss.item())
        batches += 1

    mean_loss = running_loss / max(batches, 1)
    loss_history.append(mean_loss)
    print(f"Epoch {epoch + 1}: mean training loss = {mean_loss:.6f}")

# Evaluate only on the unchanged validation fold.
aug_model.eval()
aug_val_predictions, aug_val_metrics = exp_full_pipeline(
    aug_model, val_texts, val_gold
)
aug_val_score = float(aug_val_metrics["MACRO_F1"])
aug_delta = aug_val_score - aug_reference_score

print("\n=== VALIDATION RESULTS ===")
print(f"Champion reference macro-F1: {aug_reference_score:.6f}")
print(f"Augmentation macro-F1:       {aug_val_score:.6f}")
print(f"Difference:                  {aug_delta:+.6f}")
print("\nPer-entity exact-span metrics:")
for lab in ENTITY_LABELS:
    m = aug_val_metrics[lab]
    print(
        f'{lab:12s} P={m["precision"]:.4f} '
        f'R={m["recall"]:.4f} F1={m["f1"]:.4f} '
        f'TP={m["TP"]} FP={m["FP"]} FN={m["FN"]}'
    )

# Choose a fresh filename so reruns never overwrite an earlier experiment.
out_dir = Path("/kaggle/working")
base = "experiment_entity_replacement_augmented"
checkpoint_path = out_dir / f"{base}.pt"
version = 2
while checkpoint_path.exists():
    checkpoint_path = out_dir / f"{base}_v{version}.pt"
    version += 1

torch.save(
    {
        "model_state_dict": {
            k: v.detach().cpu()
            for k, v in aug_model.state_dict().items()
        },
        "bio_labels": BIO_LABELS,
        "experiment": "entity_replacement_augmentation",
        "training_documents_including_synthetic": len(aug_texts),
        "original_training_documents": len(train_texts),
        "synthetic_documents": len(synthetic_ids),
        "epochs": AUG_EPOCHS,
        "encoder_lr": AUG_ENCODER_LR,
        "classifier_lr": AUG_CLASSIFIER_LR,
        "training_loss": loss_history,
        "validation_macro_f1": aug_val_score,
        "reference_macro_f1": aug_reference_score,
        "validation_delta": aug_delta,
        "validation_metrics": aug_val_metrics,
    },
    checkpoint_path,
)

metrics_path = checkpoint_path.with_suffix(".metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "experiment": "entity_replacement_augmentation",
            "validation_macro_f1": aug_val_score,
            "reference_macro_f1": aug_reference_score,
            "validation_delta": aug_delta,
            "training_loss": loss_history,
            "validation_metrics": aug_val_metrics,
            "test_evaluated": False,
        },
        f,
        indent=2,
    )

print("\nSaved experimental checkpoint:", checkpoint_path)
print("Saved validation metrics:", metrics_path)
print("Champion checkpoint untouched:", CHAMPION_PATH)
print("Existing prediction JSON files untouched.")
print("No test labels or test-set evaluation were used.")


In [ ]:
print("hi lol")